# Trabalho Autónomo 1

## Introdução

Em cenários de inteligência artificial, a criação de agentes autónomos capazes de navegar e interagir de forma eficiente com ambientes complexos é um desafio central e repleto de oportunidades para explorar técnicas avançadas de aprendizagem automática e tomada de decisão.

Inspirado pelo ambiente de avaliação conhecido como [*Tileworld*](https://cdn.aaai.org/AAAI/1990/AAAI90-028.pdf), este trabalho propõe a implementação de um agente autônomo projetado para operar num cenário similar.

No ambiente proposto, o agente enfrenta o desafio de recolher tijolos num terreno repleto de tijolos e buracos e tapar esses buracos com os tijolos. 

Ao desenvolver este trabalho autónomo, o objetivo é desenhar o programa que o agente correrá para concretizar o objetivo de tapar todos os buracos no terreno. O aluno deverá escrever a função **programa(agent, things, percepts)** que lhe permitirá cumprir este objetivo.  

In [28]:
'''Pacotes importados e necessários

NOTA IMPORTANTE: na mesma pasta onde está localizado este notebook jupyter devem constar os seguintes ficheiros:

- agents.py
- csp.py
- games.py
- learning.py
- logic.py
- notebook.py
- probabilistic_learning.py
- search.py
- utils.py

dependendo da instalação atual de python (ou do ambiente python) também poderão ter de ser efetuadas as instalações com 'pip install' dos seguintes pacotes:

ipythonblocks
numpy
ipywidgets
matplotlib
IPython
networkx

'''

from agents import *
from notebook import psource
from random import choice, randint

### Definição das coisas que vão existir no terreno

Primeiro vamos definir duas classes para as coisas (*Thins*) que vão existir no terreno:

A classe **Buraco**, **Parede** e a classe **Tijolo** definem três classes de objetos - **Coisas** na terminologia desta *framework* de agentes - que vão existir no **Ambiente** e com as quais o agente vai interagir.

In [29]:
class Buraco(Thing):
    pass

class Tijolo(Thing):
    pass

class Parede(Thing):
    pass

### Definição do agente que vai agir

A classe que define o nosso agente, que chamamos **Robot**, vai herdar as caracteristicas da classe **Agent** (definida no ficheiro **agent.py**) que por sua vez herda características da classe **Thing** (**Coisa**, que é definida no mesmo ficheiro).

O agente tem dois atributos:

- ***location*** o local onde se encontra nas coordenadas do ambiente.
- ***direction*** uma instância do objeto **Direction** que aponta **cima**,**baixo**,**direita** ou **esquerda**

E tem os seguintes métodos:

- **frente** que move o agente para a frente dependendo da direção a que está orientado
- **virar** que vira o agente 90º à direita ou à esquerda (só pode virar de uma vez nestes dois sentidos)
- **virar_para** que vira o agente para uma direção **cima**,**baixo**,**direita** ou **esquerda**
- **pegar** que pega numa **Coisa** caso seja da classe **Tijolo**
- **tapar** que tapa uma **Coisa** caso seja da classe **Buraco**



In [30]:
class Robot(Agent):
    location = [0,0]
    direction = Direction("down")
    
    def frente(self, success=True):
        ''' mover para a frente só apenas de suceder (i.e. local destino válido)'''
        if not success:
            return
        if self.direction.direction == Direction.R:
            self.location[0] += 1
        elif self.direction.direction == Direction.L:
            self.location[0] -= 1
        elif self.direction.direction == Direction.D:
            self.location[1] += 1
        elif self.direction.direction == Direction.U:
            self.location[1] -= 1
    
    def virar(self, d):
        '''vira o agente 90º à direita ou à esquerda (só pode virar de uma vez nestes dois sentidos)
         d deve ser do tipo Direction.R ou a string "right" ou Direction.L ou a string "left"'''
        self.direction = self.direction + d
    
    def virar_para(self, d):
        ''' vira o agente para qualquer direção (Direction.R, Direction.L, Direction.U ou Direction.D)'''
        self.direction.direction = d
        
    def pegar(self, thing):
        ''' incrementa a lista de coisas que o agente trás consigo no atributo herdado 'holding' 
        e retorna True no sucesso e False caso contrario '''
        if isinstance(thing, Tijolo):
            self.holding.append(thing)
            return True
        return False
    
    def tapar(self, thing):
        ''' Tapa um buraco com um tijolo que passa a contar menos nas coisas que trás consigo
        e retorna True no sucesso e False caso contrario'''
        if isinstance(thing, Buraco):
            self.holding = self.holding[:-1]
            return True
        return False

## Definição do terreno onde o nosso agente vai agir

A classe que define o nosso ambiente para o agente chama-se **Terreno**, esta classe herda atributos e métodos da classe ***GraphicEnvironment*** (definida no ficheiro ***agents.py***) que por sua vez herda da classe ***XYEnvironment*** que herda de ***Environment*** definidas no mesmo ficheiro.

Esta classe define cinco métodos que se destinam à simulação da ação do agente no ambiente:

- ***percept*** retorna uma lista de ***Things*** que estão ao alcance do agente na sua localição atual e atualiza essa localização.
- ***execute_action*** pede ao agente para executar uma ação em função das ações que ele é capaz.
- ***step*** para todos os agentes presentes no ambiente vai executar as ações que neles estão programadas que são função das perceções obtidas com ***percept*** nas suas atuais localizações chamando ***execute_action***.
- ***is_done*** verifica se não há mais buracos ou tijolos no terreno.
- ***run*** executa vários passos invocando ***step*** até que ***is_done*** seja verdadeira.


In [31]:
class Terreno(GraphicEnvironment):

    def percept(self, agent):
        '''retorna uma lista de coisas (things) que estão no local do nosso agente'''
        things = self.list_things_at(agent.location)
        '''faz uma copia da localização do agente'''
        loc = copy.deepcopy(agent.location)
        ''' conforme a direção do agente vai popular a variável localização'''
        if agent.direction.direction == Direction.R:
            loc[0] += 1
        elif agent.direction.direction == Direction.L:
            loc[0] -= 1
        elif agent.direction.direction == Direction.D:
            loc[1] += 1
        elif agent.direction.direction == Direction.U:
            loc[1] -= 1
        ''' se o agente saiu fora da área do terreno retorna essa perceção
        através de uma coisa chamada Bump ()
        '''
        if not self.is_inbounds(loc):
            print('{} saiu de fronteira.'.format(str(agent)[1:-1],))
            things.append(Parede())
        return things
    
    def execute_action(self, agent, action):
        '''muda o estado do ambiente com base no que o agente faz'''
        print('Executa ação ', action)

        if action == 'norte':
            print('{} decidiu virar {} no local: {}'.format(str(agent)[1:-1], action, agent.location))
            agent.virar_para(Direction.U)
            agent.frente()
        elif action == 'sul':
            print('{} decidiu virar {} no local: {}'.format(str(agent)[1:-1], action, agent.location))
            agent.virar_para(Direction.D)
            agent.frente()
        elif action == 'oeste':
            print('{} decidiu virar {} no local: {}'.format(str(agent)[1:-1], action, agent.location))
            agent.virar_para(Direction.L)
            agent.frente()
        elif action == 'este':
            print('{} decidiu virar {} no local: {}'.format(str(agent)[1:-1], action, agent.location))
            agent.virar_para(Direction.R)
            agent.frente()
        elif action == 'meia_volta':
            print('{} decidiu dar meia volta no local: {}'.format(str(agent)[1:-1], agent.location))
            agent.virar(Direction.R)
            agent.virar(Direction.R)

        elif action == 'frente':
            agent.frente()

        elif action == "pegar":
            items = self.list_things_at(agent.location, tclass=Tijolo)
            if len(items) != 0:
                if agent.pegar(items[0]):
                    self.colors[agent.__class__.__name__]=(0,int(255/(len(agent.holding)+1)),int(255/(len(agent.holding)+1)))
                    print('{} pegou {} no local: {}'.format(str(agent)[1:-1], str(items[0])[1:-1], agent.location))
                    self.delete_thing(items[0])
                    
        elif action == "tapar":
            items = self.list_things_at(agent.location, tclass=Buraco)
            if len(items) != 0:
                if agent.tapar(items[0]):
                    self.colors[agent.__class__.__name__]=(0,int(255/(len(agent.holding)+1)),int(255/(len(agent.holding)+1)))
                    print('{} tapou {} no local: {}'.format(str(agent)[1:-1], str(items[0])[1:-1], agent.location))
                    self.delete_thing(items[0])

    def step(self):
        ''' Corre o ambiente uma iteração (passo) '''
        if not self.is_done():
            for agent in self.agents:
                actions = []
                if agent.alive:
                    for a in agent.program(agent, self.things, self.percept(agent)):
                        actions.append(a)
                else:
                    actions.append("")
                for action in actions :
                    self.execute_action(agent, action)
            self.exogenous_change()
                    
    def is_done(self):
        '''Por defeito, terminamos quando já não houver buracos ou tijolos'''
        return not any(isinstance(thing, Buraco) or isinstance(thing, Tijolo) for thing in self.things)
    
    def run(self, steps=1000, delay=1):
        '''Corre o ambiente um certo número de passos, com um certo tempo de espera (delay) 
        e vai atualizando o ambiente 
        '''
        for step in range(steps):
            self.update(delay)
            if self.is_done():
                break
            self.step()
        self.update(delay)
        print('Finalizou em {} passos.'.format(step))

In [32]:
def programa(agent, things, percepts):
    '''Versão atualizada: Movimento inteligente + Evitar loops em paredes'''
    
    # 1. VERIFICAÇÃO IMEDIATA (O que está debaixo dos pés)
    for p in percepts:
        if isinstance(p, Buraco) and len(agent.holding) > 0:
            return ['tapar']
        if isinstance(p, Tijolo):
            return ['pegar']
        
    # 2. DEFINIR O ALVO (Onde queremos ir)
    # Se tenho carga, procuro o buraco mais próximo. Se não, procuro tijolo.
    alvo_classe = Buraco if len(agent.holding) > 0 else Tijolo
    
    objetivo = None
    distancia_minima = float('inf')
    
    for t in things:
        if isinstance(t, alvo_classe):
            # Cálculo de distância (Manhattan)
            dist = abs(t.location[0] - agent.location[0]) + abs(t.location[1] - agent.location[1])
            if dist < distancia_minima:
                distancia_minima = dist
                objetivo = t.location

    # 3. MOVIMENTAÇÃO ESTRATÉGICA
    if objetivo:
        dx = objetivo[0] - agent.location[0]
        dy = objetivo[1] - agent.location[1]
        
        # Decide o eixo de movimento (Prioriza X, depois Y)
        if dx > 0: return ['este']
        if dx < 0: return ['oeste']
        if dy > 0: return ['sul']
        if dy < 0: return ['norte']

    # 4. EVITAR O "VÍCIO DE FRONTEIRA" (Se não houver alvo ou bater na parede)
    # Se o agente sentir uma Parede na perceção, ele tem de mudar de direção imediatamente
    for p in percepts:
        if isinstance(p, Parede):
            # Em vez de meia-volta (que causa o loop), escolhemos uma direção aleatória que NÃO seja a atual
            opcoes = ['norte', 'sul', 'este', 'oeste']
            # Tentamos remover a direção que bateu na parede para ele não insistir
            return [choice(opcoes)]

    # 5. MODO DE EXPLORAÇÃO (Caso o mapa ainda não tenha terminado mas não vemos alvos)
    return [choice(['este', 'sul', 'oeste', 'norte'])]

In [33]:
random.seed(123456)
buraco = Buraco()
tijolo = Tijolo()
terreno = Terreno(20,20, color={'Robot': (0, 0, 255), 'Buraco': (255, 255, 255), 'Tijolo': (255, 0, 0)})
robot = Robot(programa)

terreno.add_thing(robot, [9,9])
for i in range(10):
    buraco = Buraco()
    tijolo = Tijolo()
    terreno.add_thing(buraco, [random.randint(0,terreno.width-1),random.randint(0,terreno.height-1)])
    terreno.add_thing(tijolo, [random.randint(0,terreno.width-1),random.randint(0,terreno.height-1)])
terreno.run(300,0.1)

,,,,,,,,,,,,,,,,,,,
,,,,,,,,,,,,,,,,,,,
,,,,,,,,,,,,,,,,,,,
,,,,,,,,,,,,,,,,,,,
,,,,,,,,,,,,,,,,,,,
,,,,,,,,,,,,,,,,,,,
,,,,,,,,,,,,,,,,,,,
,,,,,,,,,,,,,,,,,,,
,,,,,,,,,,,,,,,,,,,
,,,,,,,,,,,,,,,,,,,
,,,,,,,,,,,,,,,,,,,


Finalizou em 126 passos.
